<a href="https://colab.research.google.com/github/elkins/synth-pdb/blob/main/examples/interactive_tutorials/protein_quality_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [INFO] Protein Quality Assessment Lab

Welcome to the **Protein Quality Assessment Lab**!

In this tutorial, you'll learn how to evaluate protein structures like a professional structural biologist.

### [TARGET] Goal
Understand what makes a "good" vs "bad" protein structure by applying validation metrics used by the Protein Data Bank (PDB) and tools like MolProbity.

### [READ] What You'll Learn
1. **Ramachandran Analysis**: Check backbone geometry
2. **Clash Detection**: Find steric overlaps
3. **Bond Geometry**: Validate lengths and angles
4. **Rotamer Analysis**: Verify side-chain conformations
5. **Overall Quality Scoring**: Combine metrics

### [WARN] The Golden Rule
> **Garbage In, Garbage Out**
>
> A beautiful molecular dynamics simulation on a terrible structure is still terrible. Always validate your starting structures!

---

In [ ]:
# [CONFIG] SETUP: Environment and Dependencies
import os
import sys
from pathlib import Path

# Check if we are running in Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[INFO] Running in Google Colab - Installing dependencies...")
    # Increase timeout for large packages like OpenMM
    %pip install --upgrade --default-timeout=1000 -q synth-pdb biotite py3Dmol ipywidgets
else:
    print("[INFO] Running in local environment")
    # Robust path detection: find repo root by looking for 'synth_pdb' directory
    repo_root = Path(os.getcwd())
    while repo_root.parent != repo_root:
        if (repo_root / "synth_pdb").exists():
            sys.path.append(str(repo_root))
            break
        repo_root = repo_root.parent

print("[OK] Setup complete!")

In [ ]:
import io

import biotite.structure as struc
import biotite.structure.io.pdb as pdb
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from synth_pdb.generator import generate_pdb_content
from synth_pdb.validator import PDBValidator

print("[OK] All imports successful!")

## Step 1: Generate Test Structures

We'll create two structures:
- **Good Structure**: Properly minimized with realistic geometry
- **Bad Structure**: No energy minimization (raw NeRF output)

This lets us compare quality metrics side-by-side.

In [ ]:
# Generate a Good Structure (with minimization)
print("[INFO] Generating GOOD structure (with energy minimization)...")

sequence = "MKFLKFSLLTAVLLSVVFAFSSCGDDDDAK"
structure_def = "1-10:alpha,11-20:beta,21-30:random"

good_pdb = generate_pdb_content(
    sequence_str=sequence,
    structure=structure_def,
    optimize_sidechains=True,
    minimize_energy=True,  # KEY: Energy minimization ON
    minimization_max_iter=200,
)

print("[OK] Good structure ready!")
print(f"   Length: {len(good_pdb)} characters")

In [ ]:
# Generate a Bad Structure (No energy minimization + torsion drift)
print("[WARN] Generating BAD structure (Drifted Decoy)...")

# We use drift to create a structure that has a fold but is physically misfolded
bad_pdb = generate_pdb_content(
    sequence_str=sequence,
    structure=structure_def,
    drift=139.0,  # KEY: Add torsion noise to create outliers
    optimize_sidechains=False,  # KEY: No optimization to create clashes
    minimize_energy=False,  # KEY: No minimization
)

print("[OK] Bad structure (Decoy) ready!")
print(f"   Length: {len(bad_pdb)} characters")
print("\n[INFO] TIP: This 'decoy' has a realistic fold but is physically distorted.")

## Step 2: Ramachandran Plot Analysis

The **Ramachandran Plot** is the most fundamental quality check.

### [ANGLE] What is it?
It plots the backbone dihedral angles (phi, psi) for each residue. Only certain combinations are sterically allowed:
- **Blue Region**: Alpha helices (phi approx -60 degrees, psi approx -45 degrees)
- **Red Region**: Beta sheets (phi approx -120 degrees, psi approx +120 degrees)
- **Forbidden Zones**: Atoms would overlap (bad!)

### [OK] Good Structure
- 90%+ residues in favored regions
- No outliers in forbidden zones

### [FAIL] Bad Structure
- Many outliers
- Scattered distribution

In [ ]:
def plot_ramachandran(pdb_content, title="Ramachandran Plot"):
    """Extract phi/psi angles and plot Ramachandran diagram."""
    pdb_file = pdb.PDBFile.read(io.StringIO(pdb_content))
    structure = pdb_file.get_structure(model=1)

    # Calculate backbone dihedrals
    phi, psi, omega = struc.dihedral_backbone(structure)

    # Convert to degrees and filter NaN
    phi_deg = []
    psi_deg = []
    colors = []

    for i in range(len(phi)):
        if not np.isnan(phi[i]) and not np.isnan(psi[i]):
            p = np.degrees(phi[i])
            s = np.degrees(psi[i])
            phi_deg.append(p)
            psi_deg.append(s)

            # Color by region
            if -180 < p < -30 and -120 < s < 60:
                colors.append("blue")  # Alpha favored/allowed
            elif -180 < p < -45 and (60 < s < 180 or -180 < s < -140):
                colors.append("red")  # Beta favored/allowed
            elif p > 0:
                colors.append("green")  # Left-handed (usually Glycine)
            else:
                colors.append("orange")  # Other/outlier

    # Background regions
    plt.gca().add_patch(
        plt.Rectangle((-180, -120), 150, 180, color="blue", alpha=0.1, label="Alpha/Gen Allowed")
    )
    plt.gca().add_patch(
        plt.Rectangle((-180, 60), 135, 120, color="red", alpha=0.1, label="Beta Allowed")
    )
    # Wraparound for beta
    plt.gca().add_patch(plt.Rectangle((-180, -180), 135, 40, color="red", alpha=0.1))

    # Data points with small jitter
    jitter_phi = np.random.normal(0, 1.5, size=len(phi_deg))
    jitter_psi = np.random.normal(0, 1.5, size=len(psi_deg))
    plt.scatter(
        np.array(phi_deg) + jitter_phi,
        np.array(psi_deg) + jitter_psi,
        c=colors,
        alpha=0.6,
        edgecolors="black",
        s=50,
    )

    plt.xlim(-180, 180)
    plt.ylim(-180, 180)
    plt.axhline(0, color="black", linewidth=0.5, alpha=0.3)
    plt.axvline(0, color="black", linewidth=0.5, alpha=0.3)
    plt.xlabel("Phi (phi) degrees", fontsize=12)
    plt.ylabel("Psi (psi) degrees", fontsize=12)
    plt.title(title, fontsize=14, fontweight="bold")
    plt.grid(True, alpha=0.2)
    plt.legend(loc="upper right")
    plt.tight_layout()

    # Calculate statistics
    total = len(phi_deg)
    favored = sum(1 for c in colors if c in ["blue", "red"])
    outliers = sum(1 for c in colors if c == "orange")

    return {
        "total": total,
        "favored": favored,
        "favored_pct": 100 * favored / total if total > 0 else 0,
        "outliers": outliers,
        "outliers_pct": 100 * outliers / total if total > 0 else 0,
    }

In [ ]:
# Compare Good vs Bad
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

plt.sca(ax1)
good_stats = plot_ramachandran(good_pdb, title="GOOD: Minimized & Optimized")

plt.sca(ax2)
bad_stats = plot_ramachandran(bad_pdb, title="BAD: Drifted Decoy (No Minimization)")

plt.show()

# Print reports
print("[INFO] RAMACHANDRAN STATISTICS")
print("=" * 50)
print(f"GOOD Structure: {good_stats['favored_pct']:.1f}% in favored regions")
print(f"                {good_stats['outliers_pct']:.1f}% outliers")
print(f"\nBAD Structure:  {bad_stats['favored_pct']:.1f}% in favored regions")
print(f"                {bad_stats['outliers_pct']:.1f}% outliers")

print("\n[INFO] INTERPRETATION:")
print("   - 'Bad' structures (Decoys) have a fold but are physically misfolded.")
print("   - 'Good' structures have been relaxed to resolve all steric clashes.")

## Step 3: Clash Detection

**Steric clashes** occur when atoms are too close together (overlapping van der Waals radii).

### [INFO] Detection Method
For each atom pair:
1. Calculate distance
2. Compare to sum of van der Waals radii
3. If distance < 0.4 A below expected -> **CLASH!**

### [OK] Good Structure
- Few or no clashes
- Clashes only in flexible loops (acceptable)

### [FAIL] Bad Structure
- Many clashes throughout
- Clashes in core regions (unacceptable)

In [ ]:
def detect_clashes(pdb_content, clash_threshold=2.0):
    """Detect steric clashes between atoms."""
    pdb_file = pdb.PDBFile.read(io.StringIO(pdb_content))
    structure = pdb_file.get_structure(model=1)

    # Get all heavy atoms (no hydrogens)
    heavy = structure[structure.element != "H"]

    clashes = []
    n_atoms = len(heavy)

    # Pairwise distance check
    for i in range(min(n_atoms, 500)):  # Limit for speed
        for j in range(i + 1, min(n_atoms, 500)):
            if abs(heavy.res_id[i] - heavy.res_id[j]) <= 1:
                continue

            dist = np.linalg.norm(heavy.coord[i] - heavy.coord[j])

            if dist < clash_threshold:
                clashes.append(
                    {
                        "atom1": f"{heavy.res_name[i]}{heavy.res_id[i]}:{heavy.atom_name[i]}",
                        "atom2": f"{heavy.res_name[j]}{heavy.res_id[j]}:{heavy.atom_name[j]}",
                        "distance": dist,
                    }
                )

    return clashes


print("[OK] Clash detection function ready!")

In [ ]:
# Detect clashes in both structures
print("[INFO] Detecting steric clashes...\n")

good_clashes = detect_clashes(good_pdb)
bad_clashes = detect_clashes(bad_pdb)

print("[INFO] CLASH STATISTICS")
print("=" * 50)
print(f"[OK] GOOD Structure: {len(good_clashes)} clashes detected")
if good_clashes:
    print("   Top 3 clashes:")
    for clash in sorted(good_clashes, key=lambda x: x["distance"])[:3]:
        print(f"     {clash['atom1']} <-> {clash['atom2']}: {clash['distance']:.2f} A")

print(f"\n[WARN] BAD Structure: {len(bad_clashes)} clashes detected")
if bad_clashes:
    print("   Top 3 clashes:")
    for clash in sorted(bad_clashes, key=lambda x: x["distance"])[:3]:
        print(f"     {clash['atom1']} <-> {clash['atom2']}: {clash['distance']:.2f} A")

print("\n[INFO] INTERPRETATION:")
print("   <10 clashes = Excellent")
print("   10-50 clashes = Acceptable (may need refinement)")
print("   >50 clashes = Poor quality")

## Step 4: Comprehensive Validation

Now let's use synth-pdb's built-in validator to run a complete quality check.

In [ ]:
# Validate Good Structure
print("[INFO] Validating GOOD structure...\n")
good_validator = PDBValidator(good_pdb)
good_validator.validate_all()
good_violations = good_validator.get_violations()

print(f"\n{'=' * 50}")
if not good_violations:
    print("[OK] No violations! Structure passes all checks.")
else:
    print(f"[WARN] Found {len(good_violations)} violations:")
    for v in good_violations[:5]:
        print(f"   - {v}")
    if len(good_violations) > 5:
        print(f"   ... and {len(good_violations) - 5} more")

In [ ]:
# Validate Bad Structure
print("[INFO] Validating BAD structure...\n")
bad_validator = PDBValidator(bad_pdb)
bad_validator.validate_all()
bad_violations = bad_validator.get_violations()

print(f"\n{'=' * 50}")
if not bad_violations:
    print("[OK] No violations! Structure passes all checks.")
else:
    print(f"[WARN] Found {len(bad_violations)} violations:")
    for v in bad_violations[:5]:
        print(f"   - {v}")
    if len(bad_violations) > 5:
        print(f"   ... and {len(bad_violations) - 5} more")

## Step 5: Interactive Comparison

Use the widget below to toggle between good and bad structures and see the quality differences!

In [ ]:
import py3Dmol

# Create toggle widget
structure_selector = widgets.ToggleButtons(
    options=["Good Structure [OK]", "Bad Structure [WARN]"],
    description="View:",
    button_style="info",
)

output = widgets.Output()


def show_structure(change):
    with output:
        output.clear_output(wait=True)

        if "Good" in structure_selector.value:
            pdb_data = good_pdb
            title = "[OK] GOOD Structure (Minimized)"
            stats = good_stats
            clashes = len(good_clashes)
            violations = len(good_violations)
        else:
            pdb_data = bad_pdb
            title = "[WARN] BAD Structure (Not Minimized)"
            stats = bad_stats
            clashes = len(bad_clashes)
            violations = len(bad_violations)

        # 3D Viewer
        view = py3Dmol.view(width=600, height=400)
        view.addModel(pdb_data, "pdb")
        view.setStyle({"cartoon": {"color": "spectrum"}})
        view.zoomTo()
        view.show()

        # Quality Report
        print(f"\n{title}")
        print("=" * 50)
        print(f"Ramachandran favored: {stats['favored_pct']:.1f}%")
        print(f"Ramachandran outliers: {stats['outliers_pct']:.1f}%")
        print(f"Steric clashes: {clashes}")
        print(f"Total violations: {violations}")

        # Overall grade
        if clashes == 0 and stats["favored_pct"] > 30:
            print("\n[AWARD] Overall Grade: EXCELLENT (Clash-free, physically relaxed)")
        elif clashes < 5 and stats["favored_pct"] > 50:
            print("\n[OK] Overall Grade: GOOD (Minor clashes)")
        elif clashes < 15:
            print("\n[WARN] Overall Grade: ACCEPTABLE (Needs refinement)")
        else:
            print("\n[FAIL] Overall Grade: POOR (Severe steric clashes)")


structure_selector.unobserve_all(name="value")
structure_selector.observe(show_structure, names="value")

display(structure_selector, output)

with output:
    from IPython.display import HTML as _HTML
    from IPython.display import display as _disp

    _disp(
        _HTML(
            '<div style="text-align:center;padding:40px;color:#aaa;'
            "border:1px dashed #555;border-radius:8px;"
            'font-style:italic;background:#1a1a1a;">'
            "[INFO] Click a button above to load the 3D structure"
            "</div>"
        )
    )

## [AWARD] Step 6: The Scientific Defensibility Scorecard

`PDBValidator.get_quality_report()` combines five independent criteria into a single
composite verdict: **`is_overall_scientifically_defensible`**.

Each criterion maps to a peer-reviewed standard:

| Criterion | Standard | Threshold |
| :--- | :--- | :--- |
| Bond length Z-score | Engh & Huber (1991) | mean < 3.0 |
| Rotamer quality | Dunbrack library (1997) | favored > 80 % |
| Ramachandran | MolProbity / Top8000 | outliers < 5 % |
| Potential energy | OpenMM / AMBER14 | < 1 x 10^5 kJ/mol |
| Hydrophobic burial | Kauzmann (1959) Oil Drop | burial_ratio >= 0.8 |

A structure must pass **all five** to be considered scientifically defensible.


In [ ]:
def print_scorecard(label, pdb_content):
    """Run get_quality_report() and render a pass/fail scorecard."""
    v = PDBValidator(pdb_content)
    report = v.get_quality_report()

    criteria = [
        (
            "Bond Z-score < 3.0",
            report["geometric_z_scores"]["mean_bond_zscore"],
            report["geometric_z_scores"]["mean_bond_zscore"] < 3.0,
            ".3f",
        ),
        (
            "Rotamers favored > 80 %",
            report["rotamer_stats"].get("favored_rotamers_pct", 0.0),
            report["rotamer_stats"].get("favored_rotamers_pct", 0.0) > 80.0,
            ".1f",
        ),
        (
            "Ramachandran outliers < 5 %",
            report["ramachandran_stats"].get("outlier_pct", 0.0),
            report["ramachandran_stats"].get("outlier_pct", 0.0) < 5.0,
            ".1f",
        ),
        (
            "Potential energy < 1e5 kJ/mol",
            report["potential_energy_kj_mol"],
            report["is_physically_plausible"],
            ".0f",
        ),
        (
            "Burial ratio >= 0.8",
            report["hydrophobic_burial_ratio"],
            report["is_biophysically_plausible"],
            ".3f",
        ),
    ]

    sep = "=" * 54
    dash = "-" * 52
    print()
    print(sep)
    print(f"  Scorecard: {label}")
    print(sep)
    print(f"  {'Criterion':<32} {'Value':>10}  Status")
    print(f"  {'-' * 32} {'-' * 10}  ------")
    for name, value, passes, fmt in criteria:
        tick = "[OK]" if passes else "[FAIL]"
        print(f"  {name:<32} {str(format(value, fmt)):>10}  {tick}")
    print(f"  {dash}")

    verdict = report["is_overall_scientifically_defensible"]
    verdict_str = "[OK] DEFENSIBLE" if verdict else "[FAIL] NOT DEFENSIBLE"
    print(f"  Overall: {verdict_str}")
    print(sep)
    return report


good_report = print_scorecard("GOOD Structure (minimized)", good_pdb)
bad_report = print_scorecard("BAD Structure (drifted decoy)", bad_pdb)

## [GRAD] Key Takeaways

### What Makes a Good Structure?
1. **>90% Ramachandran favored** - Realistic backbone geometry
2. **<10 steric clashes** - No overlapping atoms
3. **Proper bond lengths/angles** - Chemistry makes sense
4. **Good rotamers** - Side chains in low-energy conformations

### When to Use Quality Checks
- [OK] Before starting MD simulations
- [OK] After homology modeling
- [OK] When downloading structures from PDB
- [OK] After any structure prediction/generation

### Tools for Further Analysis
- **MolProbity** - Comprehensive validation
- **PROCHECK** - Classic Ramachandran analysis
- **WHATIF** - Detailed geometry checks
- **PDB REDO** - Automated structure refinement

---

## [NEXT] Next Steps

Try these experiments:
1. Generate your own structures with different parameters
2. Download a real PDB structure and validate it
3. Compare crystal structures vs NMR structures vs AlphaFold predictions

Remember: **A validated structure is a trustworthy structure!** [INFO]